In [1]:
import json

data = json.load(open("/home/as5606/projects/Super_LlaVA_Med/model_score/gpt_judge_wa_records/openai_nano_x_ray.json"))
was = [r for r in data if r["verdict"] == "ABSTAINED"]

print(f"Wrong Abstentions: {len(was)} / {len(data)}\n")
for r in was:
    print("Q:", r["question"])
    print("A:", r["model_answer"])
    print("-" * 60)


Wrong Abstentions: 39 / 385

Q: What devices are visible in the chest x-ray?
A: On this chest X‑ray, there is a **right-sided chest tube** (a thin tube coursing along the right lateral hemithorax). No other clear implanted devices (e.g., endotracheal tube or central line) are distinctly visible in this image.
------------------------------------------------------------
Q: Is it located in the left or right lung?
A: The arrow points to an abnormal density in the **left lung** (it is on the patient’s left side—i.e., the right side of the image marker corresponds to the patient’s left).
------------------------------------------------------------
Q: What is the position of the endotracheal tube in the image?
A: The endotracheal tube tip appears to be positioned **high in the trachea—above the carina**, near the **thoracic inlet** (i.e., **not deep enough to be in the mainstem bronchi**).
------------------------------------------------------------
Q: What is the significance of the asteri

In [1]:
import json
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from open_clip import create_model_from_pretrained, get_tokenizer

# Load BioMedCLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = create_model_from_pretrained(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224', device=device
)
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
model.eval()

def get_similarity(image_path, text):
    image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    tokens = tokenizer([text]).to(device)
    with torch.inference_mode():
        img_feat = model.encode_image(image)
        txt_feat = model.encode_text(tokens)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
        sim = (img_feat * txt_feat).sum(dim=-1).item()
    return sim

# Load x_ray standard records only
data_path = "/home/as5606/projects/Super_LlaVA_Med/dataset_3/x_ray/open_ended_train_and_val/val_open_ended_annotations.jsonl"
records = [json.loads(l) for l in open(data_path) if json.loads(l)["type"] == "standard"]

q_sims, s_sims = [], []
for r in tqdm(records):  # use 100 samples for speed
    q_sims.append(get_similarity(r["image_path"], r["question"]))
    s_sims.append(get_similarity(r["image_path"], r["statement"]))

print(f"Question  + image similarity: {np.mean(q_sims):.4f} ± {np.std(q_sims):.4f}")
print(f"Statement + image similarity: {np.mean(s_sims):.4f} ± {np.std(s_sims):.4f}")
print(f"Statement wins: {sum(s > q for s, q in zip(s_sims, q_sims))}/{len(q_sims)}")


/home/as5606/miniconda3/envs/biomedclip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 385/385 [00:26<00:00, 14.63it/s]

Question  + image similarity: 0.2592 ± 0.0709
Statement + image similarity: 0.3758 ± 0.0652
Statement wins: 356/385


In [2]:
upd_records = [json.loads(l) for l in open(data_path) if json.loads(l)["type"] == "upd"]

upd_s_sims = []
for r in tqdm(upd_records):
    upd_s_sims.append(get_similarity(r["image_path"], r["statement"]))

print(f"Standard statement + image: {np.mean(s_sims):.4f} ± {np.std(s_sims):.4f}")
print(f"UPD     statement + image: {np.mean(upd_s_sims):.4f} ± {np.std(upd_s_sims):.4f}")


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 385/385 [00:07<00:00, 50.20it/s]

Standard statement + image: 0.3758 ± 0.0652
UPD     statement + image: 0.2999 ± 0.0596


In [29]:
import json, re
from collections import Counter, defaultdict
import plotly.graph_objects as go
import plotly.offline as pyo

pyo.init_notebook_mode(connected=True)

STOPWORDS = {"the", "a", "an", "is", "in", "of", "to", "and", "are", "on", "this", "what", "how",
             "does", "do", "any", "be", "it", "for", "at", "by", "can", "you", "with", "there",
             "that", "which", "was", "has", "image", "seen", "shown"}

DOMAIN_PATHS = {
    "x_ray":     "/home/as5606/projects/Super_LlaVA_Med/dataset_3/x_ray/open_ended_train_and_val/train_open_ended_annotations.jsonl",
    "mri":       "/home/as5606/projects/Super_LlaVA_Med/dataset_3/MRI/open_ended_train_and_val/train_open_ended_annotations.jsonl",
    "ct":        "/home/as5606/projects/Super_LlaVA_Med/dataset_3/CT/open_ended_train_and_val/train_open_ended_annotations.jsonl",
    "histology": "/home/as5606/projects/Super_LlaVA_Med/dataset_3/histology/open_ended_train_and_val/train_open_ended_annotations.jsonl",
    "pathology": "/home/as5606/projects/Super_LlaVA_Med/dataset_3/gross_pathology/open_ended_train_and_val/train_open_ended_annotations.jsonl",
}

def tokenize(text):
    return [w for w in re.findall(r'\b[a-z]+\b', text.lower())
            if w not in STOPWORDS and len(w) > 2]

all_questions = []
for path in DOMAIN_PATHS.values():
    records = [json.loads(l) for l in open(path)]
    all_questions += [r["question"] for r in records if r.get("type") == "standard"]

all_words = [w for q in all_questions for w in tokenize(q)]
top_inner = [w for w, _ in Counter(all_words).most_common(12)]

cooccur = defaultdict(Counter)
for q in all_questions:
    tokens = set(tokenize(q))
    for inner in top_inner:
        if inner in tokens:
            for t in tokens:
                if t != inner and t not in top_inner:
                    cooccur[inner][t] += 1

labels, parents, values = [], [], []

for inner in top_inner:
    inner_count = sum(1 for q in all_questions if inner in tokenize(q))
    labels.append(inner)
    parents.append("")
    values.append(inner_count)
    for word, count in cooccur[inner].most_common(5):
        label = f"{inner}: {word}"
        if label in labels:
            label = f"{inner}: {word}_{inner}"
        labels.append(label)
        parents.append(inner)
        values.append(count)

fig = go.Figure(go.Sunburst(
    labels=labels, parents=parents, values=values,
    textfont=dict(size=11),
))
fig.update_layout(width=700, height=700)
pyo.iplot(fig)
